# Imidazolone & Thiazolone Library Generation

Virtual combinatorial library of COX-2 selective inhibitors derived from  
commercially available aldehydes, carboxylic acids and primary amines via  
reactions *in silico*.

**Pipeline:** ```Aldehydes + Carboxylic Acids → Erlenmeyer-Plöchl → Oxazolones → Aminolysis-GFPc → Imidazolones```  

**Amines** feed into the Aminolysis-GFPc step as co-reactants.  

**Parallel branch:** ```Oxazolones → Sulphur-Exchange → Thiazolones```  

All three building-block sets pass Price → Veber filters before reactions.  
A hard global cutoff (`PriceMol <= 200.00`) is enforced in Veber filters and reaction pairing.  
Final products (Imidazolones and Thiazolones) are filtered by Veber and  
Brenk+PAINS before export, and saved as two **separate** CSV files.


In [ ]:
import gc
import pandas as pd

import py_utils as pu
from py_utils import CheckpointManager

pu.init_stage_dirs()

FORCE_RECOMPUTE = False
MAX_PRICE_MOL = 200.00  # Global cutoff for PriceMol; set to None to disable filtering

print(f"py_utils v{pu.__version__}")
print("Checkpoint system enabled")
print(f"Global PriceMol cutoff: <= {MAX_PRICE_MOL}")

py_utils v4.9
Checkpoint system enabled
Global PriceMol cutoff: <= 200.0


## 📥 1. Load SDFs

The SDFs of the compounds provided by Enamine (with their IDs) that match a
specific substructure are loaded:
- Aldehydes, R-CHO
- Carboxylic compounds, R-COOH or RCOX
- Primary amines, R-NH₂  
  
  
> SDF loading is fast — no caching needed.

In [2]:
# === CHOOSE DATASET ===
# Uncomment ONE of the two options below:

SDF_PATHS = {      # OPTION 1: Full SDF dataset (use for production runs)
    "aldehydes":   "mol_files/1. Enamine SDFs/EnamineBBStock_site_Aldehydes_12040.sdf",
    "carboxylics": "mol_files/1. Enamine SDFs/EnamineBBStock_site_CarboxylicAcids_75732.sdf",
    "amines":      "mol_files/1. Enamine SDFs/EnamineBBStock_site_PrimaryAmines_68264.sdf",
}

# SDF_PATHS = {      # OPTION 2: TESTER dataset (use for testing/validation)
#     "aldehydes":   "mol_files/1. Enamine SDFs/TESTER_Aldehydes_120.sdf",
#     "carboxylics": "mol_files/1. Enamine SDFs/TESTER_CarboxylicAcids_750.sdf",
#     "amines":      "mol_files/1. Enamine SDFs/TESTER_PrimaryAmines_680.sdf",
# }

df_aldehydes_raw   = pu.sdf_to_dataframe(SDF_PATHS["aldehydes"],   id_prefix="A") 
df_carboxylics_raw = pu.sdf_to_dataframe(SDF_PATHS["carboxylics"], id_prefix="C")
df_amines_raw      = pu.sdf_to_dataframe(SDF_PATHS["amines"],      id_prefix="N")

[SDF Reader] Loaded 12040 unique compounds from mol_files/1. Enamine SDFs/EnamineBBStock_site_Aldehydes_12040.sdf
[SDF Reader] Loaded 75732 unique compounds from mol_files/1. Enamine SDFs/EnamineBBStock_site_CarboxylicAcids_75732.sdf
[SDF Reader] Loaded 68264 unique compounds from mol_files/1. Enamine SDFs/EnamineBBStock_site_PrimaryAmines_68264.sdf


## 🔸 2. Price Filtration

Query Enamine Store API for real-time pricing. Compounds exceeding price thresholds are discarded during retrieval.

> Internal caching: prices are cached in `mol_files/2. Building Blocks/price_cache/`.

In [3]:
client = pu.EnamineClient()

print("=== Aldehydes ===")
df_aldehydes_cheap = pu.add_enamine_prices(df_aldehydes_raw, client=client)

print("\n=== Carboxylic Acids ===")
df_carboxylics_cheap = pu.add_enamine_prices(df_carboxylics_raw, client=client)

print("\n=== Amines ===")
df_amines_cheap = pu.add_enamine_prices(df_amines_raw, client=client)

pu.report_df_size(df_aldehydes_cheap,   "Aldehydes - Cheap")
pu.report_df_size(df_carboxylics_cheap, "Carboxylics - Cheap")
pu.report_df_size(df_amines_cheap,      "Amines - Cheap")

=== Aldehydes ===
[Enamine Pricing] Processing 12040 compounds...
[Enamine Pricing] Querying 12040 compounds via API
[Enamine Pricing] Filters: <= 40.0 EUR/g, <= 250.0 EUR/pack
[Enamine Pricing] Batch 1/7: 2000 compounds
[EnamineClient] Signed in successfully.
[Enamine Pricing] Batch 2/7: 2000 compounds
[Enamine Pricing] Batch 3/7: 2000 compounds
[Enamine Pricing] Batch 4/7: 2000 compounds
[Enamine Pricing] Batch 5/7: 2000 compounds
[Enamine Pricing] Batch 6/7: 2000 compounds
[Enamine Pricing] Batch 7/7: 40 compounds
[Enamine Pricing] Saved 1570 valid + 10470 invalid to cache
[Enamine Pricing] Completed: 1569/12040 with valid pricing
⚠️  Removed 10471 compounds (no valid pricing)

=== Carboxylic Acids ===
[Enamine Pricing] Loaded 1570 valid + 10470 invalid cached
[Enamine Pricing] Processing 75732 compounds...
[Enamine Pricing] Querying 75732 compounds via API
[Enamine Pricing] Filters: <= 40.0 EUR/g, <= 250.0 EUR/pack
[Enamine Pricing] Batch 1/38: 2000 compounds
[Enamine Pricing] Batc

## 🔸 3. Veber filtering (of building blocks)

Compute RDKit molecular descriptors, then apply per-class bioavailability criteria.  

Each building block class uses independent limits (`VEBER_ALDEHYDES`, `VEBER_CARBOXYLICS`,`VEBER_AMINES`) back-calculated  
from the final product thresholds using each reaction's property increments (ΔtPSA, ΔRotB, ΔLogP, ΔMW, ΔHBD, ΔHBA).  

| Reaction | ΔtPSA | ΔRotB | ΔLogP | ΔMW | ΔHBD | ΔHBA |
|----------|------|-------|--------|------|------|-----|
| E-P      | -15.71 | 0 | 0.4127 | +21.022 | -1 | +1 |
| A-G      | -32.01 | 1 | 0.3403 | -18.015 | -1 | -2 |
| S-E      | +16.07 | 0 | +0.7166 | +16.068 | -1 | 0 |

In [4]:
VEBER_BB = {
    "aldehydes":   dict(max_tPSA=118.41, max_RotB=10, max_LogP=4.4964, max_MWT=405.883, max_HBD=5, max_HBA=9),
    "carboxylics": dict(max_tPSA=138.64, max_RotB=10, max_LogP=4.3821, max_MWT=421.882, max_HBD=6, max_HBA=9),
    "amines":      dict(max_tPSA=133.35, max_RotB=9,  max_LogP=3.7943, max_MWT=377.873, max_HBD=6, max_HBA=9),
}

BB_ID_COLS = ["ID", "Catalog_ID", "SMILES", "PriceMol", "PriceG", "tPSA", "RotB", "LogP", "MW", "HBD", "HBA"]
BB_REJ_COLS = BB_ID_COLS + ["Rejection"]
PRODUCT_RAW_COLS = ["ID", "SMILES", "PriceMol"]

def _veber_bb_filter(df: pd.DataFrame, veber_params: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    return pu.filter_Veber(
        pu.add_rdkit_properties(df),
        max_price_mol=MAX_PRICE_MOL,
        **veber_params,
    )

df_aldehydes_druglike, _ = pu.load_or_filter(
    lambda: _veber_bb_filter(df_aldehydes_cheap, VEBER_BB["aldehydes"]),
    accepted_csv=pu.stage_path("Aldehydes", filter_mode="veber"),
    rejected_csv=pu.rejected_path("Aldehydes", "veber"),
    force_recompute=FORCE_RECOMPUTE,
    input_row_count=len(df_aldehydes_cheap),
    params={"max_price_mol": MAX_PRICE_MOL, "veber": VEBER_BB["aldehydes"]},
)

df_carboxylics_druglike, _ = pu.load_or_filter(
    lambda: _veber_bb_filter(df_carboxylics_cheap, VEBER_BB["carboxylics"]),
    accepted_csv=pu.stage_path("Carboxylics", filter_mode="veber"),
    rejected_csv=pu.rejected_path("Carboxylics", "veber"),
    force_recompute=FORCE_RECOMPUTE,
    input_row_count=len(df_carboxylics_cheap),
    params={"max_price_mol": MAX_PRICE_MOL, "veber": VEBER_BB["carboxylics"]},
)

df_amines_druglike, _ = pu.load_or_filter(
    lambda: _veber_bb_filter(df_amines_cheap, VEBER_BB["amines"]),
    accepted_csv=pu.stage_path("Amines", filter_mode="veber"),
    rejected_csv=pu.rejected_path("Amines", "veber"),
    force_recompute=FORCE_RECOMPUTE,
    input_row_count=len(df_amines_cheap),
    params={"max_price_mol": MAX_PRICE_MOL, "veber": VEBER_BB["amines"]},
)

pu.report_df_size(df_aldehydes_druglike,   "Aldehydes - Druglike")
pu.report_df_size(df_carboxylics_druglike, "Carboxylics - Druglike")
pu.report_df_size(df_amines_druglike,      "Amines - Druglike")

[load_or_filter] No cache pair matches input row count (1,569); recomputing
[load_or_filter] Computing Aldehydes_veber.csv...
[Veber] Processing 1,569 molecules in 1 chunks...
[filter_Veber] 278/1,569 accepted (17.7%), 1,291 rejected
[filter_Veber] Cache: 0 hits, 3138 misses (0.0% hit rate)
[load_or_filter] Saved Aldehydes_veber_278cmpds.csv (278 accepted) + Aldehydes_rejected_veber_1291cmpds.csv (1,291 rejected)
[load_or_filter] No cache pair matches input row count (4,964); recomputing
[load_or_filter] Computing Carboxylics_veber.csv...
[Veber] Processing 4,964 molecules in 1 chunks...
[filter_Veber] 745/4,964 accepted (15.0%), 4,219 rejected
[filter_Veber] Cache: 4 hits, 9924 misses (0.0% hit rate)
[load_or_filter] Saved Carboxylics_veber_745cmpds.csv (745 accepted) + Carboxylics_rejected_veber_4219cmpds.csv (4,219 rejected)
[load_or_filter] No cache pair matches input row count (5,360); recomputing
[load_or_filter] Computing Amines_veber.csv...
[Veber] Processing 5,360 molecules in

## 🔹 4. Erlenmeyer-Plöchl Reaction

Aldehydes + Carboxylic Acids → Oxazolones

In [5]:
# E-P reaction outputs to {Stage}_raw.csv (not {Stage}.csv)
df_oxazolones_raw = pu.load_or_run(
    lambda checkpoint_manager: pu.rxn_ErlenmeyerPlochl(
        df_aldehydes_druglike, 
        df_carboxylics_druglike,
        chunk_size=25,
        checkpoint_manager=checkpoint_manager,
        max_price_mol=MAX_PRICE_MOL,
    ),
    output_csv=pu.stage_path("Oxazolones", filter_mode="raw"),
    stage_name="Oxazolones",
    force_recompute=FORCE_RECOMPUTE,
    params={"max_price_mol": MAX_PRICE_MOL, "chunk_size": 25},
)

pu.report_df_size(df_oxazolones_raw, "Oxazolones - Raw")

[load_or_run] Resuming from checkpoint: 0/0 chunks completed
[ErlenmeyerPlochl] Processing chunk 1/12 (25 aldehydes × 745 carboxylics)
[ErlenmeyerPlochl] 207,110 total pairs (12 chunks of ~25 aldehydes)
[ErlenmeyerPlochl] Processing chunk 2/12 (25 aldehydes × 745 carboxylics)
[ErlenmeyerPlochl] Processing chunk 3/12 (25 aldehydes × 745 carboxylics)
[ErlenmeyerPlochl] Processing chunk 4/12 (25 aldehydes × 745 carboxylics)
[ErlenmeyerPlochl] Processing chunk 5/12 (25 aldehydes × 745 carboxylics)
[ErlenmeyerPlochl] Processing chunk 6/12 (25 aldehydes × 745 carboxylics)
[ErlenmeyerPlochl] Processing chunk 7/12 (25 aldehydes × 745 carboxylics)
[ErlenmeyerPlochl] Processing chunk 8/12 (25 aldehydes × 745 carboxylics)
[ErlenmeyerPlochl] Processing chunk 9/12 (25 aldehydes × 745 carboxylics)
[ErlenmeyerPlochl] Processing chunk 10/12 (25 aldehydes × 745 carboxylics)
[ErlenmeyerPlochl] Processing chunk 11/12 (25 aldehydes × 745 carboxylics)
[ErlenmeyerPlochl] Processing chunk 12/12 (3 aldehydes 

## 🔸 5. Veber Filter (Oxazolones)

Apply bioavailability criteria to oxazolone intermediates.
Limits are set to the most permissive value across both downstream pathways (A-G and S-E)
to avoid discarding valid precursors prematurely.

In [6]:
VEBER_OXAZOLONES = dict(
    max_tPSA=145.99, max_RotB=10, max_LogP=5.0848,
    max_MWT=486.957, max_HBD=5, max_HBA=11
)

# Filter output: {Stage}_{N}cmpds.csv (row count in filename)
df_oxazolones_druglike, _ = pu.load_or_filter(
    lambda: pu.filter_Veber(
        pu.add_rdkit_properties(df_oxazolones_raw),
        max_price_mol=MAX_PRICE_MOL,
        **VEBER_OXAZOLONES,
    ),
    accepted_csv=pu.stage_path("Oxazolones", filter_mode="veber"),
    rejected_csv=pu.rejected_path("Oxazolones", "veber"),
    force_recompute=FORCE_RECOMPUTE,
    input_row_count=len(df_oxazolones_raw),
    params={"max_price_mol": MAX_PRICE_MOL, "veber": VEBER_OXAZOLONES},
)

pu.report_df_size(df_oxazolones_druglike, "Oxazolones - Druglike")

[load_or_filter] No cache pair matches input row count (81,351); recomputing
[load_or_filter] Computing Oxazolones_veber.csv...
[Veber] Processing 81,351 molecules in 9 chunks...
[filter_Veber] 72,831/81,351 accepted (89.5%), 8,520 rejected
[filter_Veber] Cache: 0 hits, 162702 misses (0.0% hit rate)
[load_or_filter] Saved Oxazolones_veber_72831cmpds.csv (72,831 accepted) + Oxazolones_rejected_veber_8520cmpds.csv (8,520 rejected)
[Oxazolones - Druglike] 72831 rows


## 🔹 6. Aminolysis-GFPc Reaction

Oxazolones + Amines → Imidazolones

> ⚠️ This reaction generates billions of products. Checkpointing is enabled by default.

In [7]:
# Free memory — DataFrames no longer needed after E-P and Veber
del df_oxazolones_raw
del df_aldehydes_raw, df_carboxylics_raw, df_amines_raw
del df_aldehydes_cheap, df_carboxylics_cheap, df_amines_cheap
del df_aldehydes_druglike, df_carboxylics_druglike
gc.collect()
print("[Memory] Freed unnecessary DataFrames before A-G")

# A-G reaction outputs to {Stage}_raw.csv
df_imidazolones_raw = pu.load_or_run(
    lambda checkpoint_manager: pu.rxn_AminolysisGFPc(
        df_oxazolones_druglike, 
        df_amines_druglike,
        chunk_size=10,
        checkpoint_manager=checkpoint_manager,
        max_price_mol=MAX_PRICE_MOL,
    ),
    output_csv=pu.stage_path("Imidazolones", filter_mode="raw"),
    stage_name="Imidazolones",
    force_recompute=FORCE_RECOMPUTE,
    params={"max_price_mol": MAX_PRICE_MOL, "chunk_size": 10},
)

df_imidazolones_raw = df_imidazolones_raw[PRODUCT_RAW_COLS].copy()

pu.report_df_size(df_imidazolones_raw, "Imidazolones - Raw")

[Memory] Freed unnecessary DataFrames before A-G
[load_or_run] Resuming from checkpoint: 0/0 chunks completed
[AminolysisGFPc] 60,886,716 total pairs (7284 chunks of ~10 oxazolones)
[AminolysisGFPc] Processing chunk 500/7284 (4,990 oxazolones processed)
[AminolysisGFPc] Processing chunk 1000/7284 (9,990 oxazolones processed)
[AminolysisGFPc] Processing chunk 1500/7284 (14,990 oxazolones processed)
[AminolysisGFPc] Processing chunk 2000/7284 (19,990 oxazolones processed)
[AminolysisGFPc] Processing chunk 2500/7284 (24,990 oxazolones processed)
[AminolysisGFPc] Processing chunk 3000/7284 (29,990 oxazolones processed)
[AminolysisGFPc] Processing chunk 3500/7284 (34,990 oxazolones processed)
[AminolysisGFPc] Processing chunk 4000/7284 (39,990 oxazolones processed)
[AminolysisGFPc] Processing chunk 4500/7284 (44,990 oxazolones processed)
[AminolysisGFPc] Processing chunk 5000/7284 (49,990 oxazolones processed)
[AminolysisGFPc] Processing chunk 5500/7284 (54,990 oxazolones processed)
[Aminol

## 🔹 7. Sulphur-Exchange Reaction

Oxazolones → Thiazolones  

> **NB:** input is `df_oxazolones_druglike` (post-Veber, Cell 5) — not `df_imidazolones_raw` or `df_oxazolones_raw`.

In [8]:
THIOACETIC_PRICE_EQ = 10.0

# S-E reaction outputs to {Stage}_raw.csv
df_thiazolones_raw = pu.load_or_run(
    lambda checkpoint_manager: pu.rxn_SulphurExchange(
        df_oxazolones_druglike, 
        thioacetic_price_eq=THIOACETIC_PRICE_EQ,
        chunk_size=10000,
        checkpoint_manager=checkpoint_manager,
        max_price_mol=MAX_PRICE_MOL,
    ),
    output_csv=pu.stage_path("Thiazolones", filter_mode="raw"),
    stage_name="Thiazolones",
    force_recompute=FORCE_RECOMPUTE,
    params={"max_price_mol": MAX_PRICE_MOL, "chunk_size": 10000, "thioacetic_price_eq": THIOACETIC_PRICE_EQ},
)

df_thiazolones_raw = df_thiazolones_raw[PRODUCT_RAW_COLS].copy()

pu.report_df_size(df_thiazolones_raw, "Thiazolones - Raw")

[load_or_run] Resuming from checkpoint: 0/0 chunks completed
[SulphurExchange] 72,831 valid oxazolones: 0 cache hits, 61,874 misses
[SulphurExchange] Processing chunk 1/7 (10,000 items)
[SulphurExchange] Processing chunk 2/7 (10,000 items)
[SulphurExchange] Processing chunk 3/7 (10,000 items)
[SulphurExchange] Processing chunk 4/7 (10,000 items)
[SulphurExchange] Processing chunk 5/7 (10,000 items)
[SulphurExchange] Processing chunk 6/7 (10,000 items)
[SulphurExchange] Processing chunk 7/7 (1,874 items)
[SulphurExchange] Checkpoint saved: Thiazolones_checkpoint.json
[SulphurExchange] Cache: 0 hits, 61874 misses (0.0% hit rate)
[SulphurExchange] Stats: {'input_oxazolones': 72831, 'thioacetic_price_eq': 10.0, 'invalid_oxazolone': 0, 'not_oxazolone': 0, 'skipped_price': 10957, 'no_product': 0, 'problematic': 0, 'output_rows': 61874, 'cache_hits': 0, 'cache_misses': 61874}
[load_or_run] Saved Thiazolones_raw_61874cmpds.csv (61,874 rows)
[Thiazolones - Raw] 61874 rows


## 🔸 8. Veber Filter (Products)

Drop any stale descriptor columns, recompute product descriptors, and apply Veber criteria to both product families.  

`VEBER_PRODUCTS` defines the final target thresholds shared by imidazolones and thiazolones.

In [9]:
VEBER_PRODUCTS = dict(
    max_tPSA=140, max_RotB=10, max_LogP=5,
    max_MWT=500, max_HBD=5, max_HBA=10
)

# Filter outputs: {Stage}_{N}cmpds.csv (row count in filename)
df_imidazolones_druglike, _ = pu.load_or_filter(
    lambda: pu.filter_Veber(
        df_imidazolones_raw,
        recompute_descriptors=True,
        max_price_mol=MAX_PRICE_MOL,
        **VEBER_PRODUCTS,
    ),
    accepted_csv=pu.stage_path("Imidazolones", filter_mode="veber"),
    rejected_csv=pu.rejected_path("Imidazolones", "veber"),
    force_recompute=FORCE_RECOMPUTE,
    input_row_count=len(df_imidazolones_raw),
    params={"max_price_mol": MAX_PRICE_MOL, "veber": VEBER_PRODUCTS, "recompute_descriptors": True},
)

df_thiazolones_druglike, _ = pu.load_or_filter(
    lambda: pu.filter_Veber(
        df_thiazolones_raw,
        recompute_descriptors=True,
        max_price_mol=MAX_PRICE_MOL,
        **VEBER_PRODUCTS,
    ),
    accepted_csv=pu.stage_path("Thiazolones", filter_mode="veber"),
    rejected_csv=pu.rejected_path("Thiazolones", "veber"),
    force_recompute=FORCE_RECOMPUTE,
    input_row_count=len(df_thiazolones_raw),
    params={"max_price_mol": MAX_PRICE_MOL, "veber": VEBER_PRODUCTS, "recompute_descriptors": True},
)

pu.report_df_size(df_imidazolones_druglike, "Imidazolones - Druglike")
pu.report_df_size(df_thiazolones_druglike, "Thiazolones - Druglike")

[load_or_filter] No cache pair matches input row count (4,054,102); recomputing
[load_or_filter] Computing Imidazolones_veber.csv...
[Veber] Processing 4,054,102 molecules in 406 chunks...
[Veber] Processing chunk 100/406...
[Veber] Processing chunk 200/406...
[Veber] Processing chunk 300/406...
[Veber] Processing chunk 400/406...
[filter_Veber] 3,049,994/4,054,102 accepted (75.2%), 1,004,108 rejected
[filter_Veber] Cache: 0 hits, 8108204 misses (0.0% hit rate)
[load_or_filter] Saved Imidazolones_veber_3049994cmpds.csv (3,049,994 accepted) + Imidazolones_rejected_veber_1004108cmpds.csv (1,004,108 rejected)
[load_or_filter] No cache pair matches input row count (61,874); recomputing
[load_or_filter] Computing Thiazolones_veber.csv...
[Veber] Processing 61,874 molecules in 7 chunks...
[filter_Veber] 49,087/61,874 accepted (79.3%), 12,787 rejected
[filter_Veber] Cache: 0 hits, 123748 misses (0.0% hit rate)
[load_or_filter] Saved Thiazolones_veber_49087cmpds.csv (49,087 accepted) + Thiazol

## 🔸 9. Brenk + PAINS Filter

Remove compounds containing Brenk structural alerts or PAINS substructures from products.

In [10]:
df_imidazolones_untoxic, _ = pu.load_or_filter(
    lambda: pu.filter_BrenkPAINS(df_imidazolones_druglike),
    accepted_csv=pu.stage_path("Imidazolones", filter_mode="brenkpains"),
    rejected_csv=pu.rejected_path("Imidazolones", "brenkpains"),
    force_recompute=FORCE_RECOMPUTE,
    input_row_count=len(df_imidazolones_druglike),
    params={"filter": "brenkpains", "max_price_mol": MAX_PRICE_MOL},
)

df_thiazolones_untoxic, _ = pu.load_or_filter(
    lambda: pu.filter_BrenkPAINS(df_thiazolones_druglike),
    accepted_csv=pu.stage_path("Thiazolones", filter_mode="brenkpains"),
    rejected_csv=pu.rejected_path("Thiazolones", "brenkpains"),
    force_recompute=FORCE_RECOMPUTE,
    input_row_count=len(df_thiazolones_druglike),
    params={"filter": "brenkpains", "max_price_mol": MAX_PRICE_MOL},
)

pu.report_df_size(df_imidazolones_untoxic, "Imidazolones - Untoxic")
pu.report_df_size(df_thiazolones_untoxic, "Thiazolones - Untoxic")

[load_or_filter] No cache pair matches input row count (3,049,994); recomputing
[load_or_filter] Computing Imidazolones_brenkpains.csv...
[filter_BrenkPAINS] 1868308/3049994 accepted (61.3%)
[filter_BrenkPAINS] Rejected: Brenk=1156866, PAINS=24820
[load_or_filter] Saved Imidazolones_brenkpains_1868308cmpds.csv (1,868,308 accepted) + Imidazolones_rejected_brenkpains_1181686cmpds.csv (1,181,686 rejected)
[load_or_filter] No cache pair matches input row count (49,087); recomputing
[load_or_filter] Computing Thiazolones_brenkpains.csv...
[filter_BrenkPAINS] 32073/49087 accepted (65.3%)
[filter_BrenkPAINS] Rejected: Brenk=16469, PAINS=545
[load_or_filter] Saved Thiazolones_brenkpains_32073cmpds.csv (32,073 accepted) + Thiazolones_rejected_brenkpains_17014cmpds.csv (17,014 rejected)
[Imidazolones - Untoxic] 1868308 rows
[Thiazolones - Untoxic] 32073 rows


## 📤 10. Export

Imidazolones and Thiazolones saved as **separate** CSV files.

In [11]:
PROD_COLS = ["ID", "SMILES", "PriceMol", "tPSA", "RotB", "LogP", "MW", "HBD", "HBA", "MR", "HvyAtm", "Rings", "CAtm", "HetAtm"]

# Final exports: {Stage}_{N}cmpds.csv (includes row count for clarity)
n_imi = len(df_imidazolones_untoxic)
n_thi = len(df_thiazolones_untoxic)

pu.save_dataframe(df_imidazolones_untoxic[PROD_COLS], 
                  pu.stage_path("Imidazolones", row_count=n_imi, filter_mode="brenkpains"))
pu.save_dataframe(df_thiazolones_untoxic[PROD_COLS],
                  pu.stage_path("Thiazolones", row_count=n_thi, filter_mode="brenkpains"))

print("\n=== Pipeline Complete ===")
print(f"Imidazolones: {n_imi:,} compounds")
print(f"Thiazolones:  {n_thi:,} compounds")


[save_dataframe] Saved Imidazolones_brenkpains_1868308cmpds.csv (1,868,308 rows)
[save_dataframe] Saved Thiazolones_brenkpains_32073cmpds.csv (32,073 rows)

=== Pipeline Complete ===
Imidazolones: 1,868,308 compounds
Thiazolones:  32,073 compounds
